<a href="https://colab.research.google.com/github/peterbabulik/QuantumWalker/blob/main/QGF_QEC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit cma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 5.4 MB/s eta 0:00:00


In [2]:

# ==============================================================================
#  PREAMble: IMPORTS AND SETUP
# ==============================================================================
import numpy as np
import time
import warnings

# Qiskit Imports
import qiskit
from qiskit.circuit import QuantumCircuit, Gate
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.synthesis.two_qubit import TwoQubitBasisDecomposer
from qiskit.circuit.library import CXGate, CCXGate

# The powerful "Designer AI" engine
import cma

# Scipy for the Forge
from scipy.linalg import expm

# Suppress benign warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Qiskit version: {qiskit.__version__}")
print("\n--- The QEC Gate Forge: Evolving an Error Correction Gate ---")

# ==============================================================================
#  CORE FORGE ENGINE (UNCHANGED)
# ==============================================================================

PAULI_BASIS_2Q = [p1+p2 for p1 in 'IXYZ' for p2 in 'IXYZ' if p1+p2 != 'II']

def build_gate_from_coeffs(coeffs: np.ndarray, decomposer: TwoQubitBasisDecomposer) -> Gate:
    """Builds a unitary matrix and synthesizes it into a Qiskit Gate."""
    generator_h = SparsePauliOp(PAULI_BASIS_2Q, coeffs=coeffs)
    u_matrix = expm(-1j * generator_h.to_matrix())
    forged_gate = Gate(name="GUARDIAN_U", num_qubits=2, params=[])
    forged_gate.definition = decomposer(u_matrix)
    return forged_gate

# ==============================================================================
#  THE QEC FORGE EXPERIMENT
# ==============================================================================
if __name__ == "__main__":

    NUM_DATA_QUBITS = 3
    NUM_ANCILLA = 2
    TOTAL_QUBITS = NUM_DATA_QUBITS + NUM_ANCILLA

    # Define the target outcomes for our fitness function
    # The key is the desired ancilla state, the value is the input state
    QEC_TARGETS = {
        "00": "000", # No error
        "01": "100", # Error on qubit 0
        "10": "010", # Error on qubit 1
        "11": "001"  # Error on qubit 2
    }

    # We need a decomposer for the fitness function
    decomposer = TwoQubitBasisDecomposer(CXGate())

    # --- The Guardian Fitness Function ---
    # This is the "test drive" that evaluates how good a forged gate is at QEC.
    def qec_fitness_function(coeffs: np.ndarray) -> float:
        try:
            # Forge the gate for this evaluation
            guardian_gate = build_gate_from_coeffs(coeffs, decomposer)

            # Build the syndrome measurement circuit using the guardian gate
            # This is a generic structure we are asking the AI to make work.
            qc = QuantumCircuit(TOTAL_QUBITS)
            qc.append(guardian_gate, [0, 3]) # Compare qubit 0 and 1 via ancilla 0
            qc.append(guardian_gate, [1, 3])
            qc.append(guardian_gate, [1, 4]) # Compare qubit 1 and 2 via ancilla 1
            qc.append(guardian_gate, [2, 4])

            total_success_prob = 0
            for ancilla_target, data_input in QEC_TARGETS.items():

                # Prepare the initial state (e.g., |100>|00>)
                initial_state_str = data_input + "00"
                initial_state_idx = int(initial_state_str, 2)
                initial_state_vec = np.zeros(2**TOTAL_QUBITS)
                initial_state_vec[initial_state_idx] = 1.0

                # Simulate the circuit
                final_state = Statevector(initial_state_vec).evolve(qc)

                # Calculate the probability of measuring the correct ancilla state
                probs = final_state.probabilities_dict()

                correct_prob_for_case = 0
                for outcome, prob in probs.items():
                    # Qiskit bit order is reversed, so ancillas are at the end
                    measured_ancilla = outcome[:NUM_ANCILLA]
                    if measured_ancilla == ancilla_target:
                        correct_prob_for_case += prob

                total_success_prob += correct_prob_for_case

            # Average success probability across all 4 cases
            avg_success_prob = total_success_prob / 4.0

            # CMA-ES minimizes, so return 1 - success
            return 1.0 - avg_success_prob

        except Exception as e:
            # print(f"Error: {e}") # for debugging
            return 1.0 # Bad fitness on failure

    # --- Set up and run the CMA-ES Optimizer ---
    print("--- Starting AI Forge to Evolve a QEC Gate ---")
    x0 = np.random.rand(15) * 2 * np.pi - np.pi
    sigma0 = 0.5
    options = {'bounds': [-np.pi, np.pi], 'maxfevals': 1000, 'verbose': -9}

    master_es = cma.CMAEvolutionStrategy(x0, sigma0, options)

    start_time = time.time()
    master_es.optimize(qec_fitness_function)
    end_time = time.time()

    print(f"  > Forge complete in {end_time - start_time:.2f}s.")

    # --- Retrieve the Champion Guardian Gate ---
    champion_coeffs = master_es.result.xbest
    champion_fitness = master_es.result.fbest
    champion_avg_success = 1.0 - champion_fitness

    # ==============================================================================
    #  THE GAUNTLET: AI vs. HUMAN
    # ==============================================================================
    print("\n--- THE GAUNTLET: AI-Forged Gate vs. Human-Designed Circuit ---")

    # --- Build the Human-Designed QEC Circuit ---
    # The standard textbook implementation using CNOTs and a Toffoli
    human_qc = QuantumCircuit(TOTAL_QUBITS, name="Human_QEC")
    human_qc.cx(0, 3)
    human_qc.cx(1, 3)
    human_qc.cx(1, 4)
    human_qc.cx(2, 4)
    # The commented out Toffoli is for a different code, this is sufficient for bit-flip
    # human_qc.ccx(3, 4, 2) # Example of more complex logic

    # --- Build the AI-Designed QEC Circuit ---
    ai_champion_gate = build_gate_from_coeffs(champion_coeffs, decomposer)
    ai_qc = QuantumCircuit(TOTAL_QUBITS, name="AI_QEC")
    ai_qc.append(ai_champion_gate, [0, 3])
    ai_qc.append(ai_champion_gate, [1, 3])
    ai_qc.append(ai_champion_gate, [1, 4])
    ai_qc.append(ai_champion_gate, [2, 4])

    # --- Evaluate Both ---
    print("  > Evaluating performance of both circuits...")
    circuits_to_test = {"Human": human_qc, "AI": ai_qc}
    results = {}

    for name, qc in circuits_to_test.items():
        total_success_prob = 0
        for ancilla_target, data_input in QEC_TARGETS.items():
            initial_state_str = data_input + "00"
            initial_state_idx = int(initial_state_str, 2)
            initial_state_vec = np.zeros(2**TOTAL_QUBITS); initial_state_vec[initial_state_idx] = 1.0
            final_state = Statevector(initial_state_vec).evolve(qc)
            probs = final_state.probabilities_dict()
            correct_prob_for_case = 0
            for outcome, prob in probs.items():
                if outcome[:NUM_ANCILLA] == ancilla_target:
                    correct_prob_for_case += prob
            total_success_prob += correct_prob_for_case
        results[name] = total_success_prob / 4.0

    human_perf = results["Human"]
    ai_perf = results["AI"]

    # ==============================================================================
    #  FINAL VERDICT
    # ==============================================================================
    print("\n--- FINAL VERDICT ---")
    print(f"Human-Designed Circuit Avg. Success: {human_perf:.2%}")
    print(f"AI-Forged Circuit Avg. Success:      {ai_perf:.2%}")

    print("\n--- Analysis of the Champion Guardian Gate ---")
    print("Its generator H = sum(c_i * P_i) was defined by:")
    sorted_coeffs = sorted(zip(PAULI_BASIS_2Q, champion_coeffs), key=lambda item: abs(item[1]), reverse=True)
    for pauli, coeff in sorted_coeffs[:5]:
        print(f"  {pauli}: {coeff:.3f}")

    print("\n--- Final Conclusion ---")
    if ai_perf > human_perf * 1.001:
        print(">>> A NEW GUARDIAN IS BORN! <<<")
        print("The AI discovered a gate that builds a MORE effective QEC circuit than the standard design.")
        print("This could be a significant discovery for building more efficient error correction codes.")
    elif human_perf > ai_perf * 1.001:
        print(">>> THE TEXTBOOK DESIGN REMAINS SUPERIOR. <<<")
        print("The standard human-designed circuit is more effective. The AI's gate, while optimized")
        print("for the generic structure, could not beat the elegant logic of the CNOT-based circuit.")
    else:
        print(">>> A TIE! AN ALTERNATIVE PATH DISCOVERED. <<<")
        print("The AI found a completely different gate that performs identically to the human design.")
        print("This is a success, demonstrating there are multiple ways to build effective QEC codes.")

Qiskit version: 2.1.1

--- The QEC Gate Forge: Evolving an Error Correction Gate ---
--- Starting AI Forge to Evolve a QEC Gate ---
  > Forge complete in 48.55s.

--- THE GAUNTLET: AI-Forged Gate vs. Human-Designed Circuit ---
  > Evaluating performance of both circuits...

--- FINAL VERDICT ---
Human-Designed Circuit Avg. Success: 25.00%
AI-Forged Circuit Avg. Success:      73.08%

--- Analysis of the Champion Guardian Gate ---
Its generator H = sum(c_i * P_i) was defined by:
  IZ: -3.130
  XY: 3.128
  YY: 3.086
  YI: 2.290
  XI: 2.212

--- Final Conclusion ---
>>> A NEW GUARDIAN IS BORN! <<<
The AI discovered a gate that builds a MORE effective QEC circuit than the standard design.
This could be a significant discovery for building more efficient error correction codes.
